In [ ]:
!pip install -q executorch torchvision huggingface_hub pytorch-tokenizers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
from huggingface_hub import login, snapshot_download

login() # paste your Hugging Face token in the box that appears
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"],
 local_dir="/content/waste-assistant/llama-1b")

# List contents using Python instead of Bash
print(os.listdir("/content/waste-assistant/llama-1b/original"))
# you must see: consolidated.00.pth params.json tokenizer.model

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

['params.json', 'consolidated.00.pth', 'tokenizer.model']


In [ ]:
%%writefile /content/waste-assistant/export_config.yaml
base:
  model_class: llama3_2
  checkpoint: /content/waste-assistant/llama-1b/original/consolidated.00.pth
  params: /content/waste-assistant/llama-1b/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w

Overwriting /content/waste-assistant/export_config.yaml


In [ ]:
%%bash
cd /content/waste-assistant
python -m executorch.extension.llm.export.export_llm --config export_config.yaml \
 +export.output_name="llama3_2_1b.pte" \
 +export.max_seq_length=1024 +export.max_context_length=1024 \
 model.use_sdpa_with_kv_cache=False
ls -lh *.pte # expect a file of roughly 1 GB

[2026-10-30 07:42:26,531][root][INFO] - Applying quantizers: []
[2026-10-30 07:42:26,661][root][INFO] - Checkpoint dtype: torch.bfloat16
quantize ('tok_embeddings', Embedding(128256, 2048)) with group_size 32, bitwidth 4
[2026-10-30 07:49:06,422][root][INFO] - Model after source transforms: Transformer(
  (tok_embeddings): QuantizedGroupEmbedding()
  (layers): ModuleList(
    (0-15): 16 x TransformerBlock(
      (attention): AttentionMHA(
        (wq): Linear(in_features=2048, out_features=2048, bias=False, weight=IntxUnpackedToInt8Tensor(target_dtype=torch.int4, block_size=(1, 128), shape=torch.Size([2048, 2048]), dtype=torch.float32, device=cpu, activation_quantization=int8_asym_per_token))
        (wk): Linear(in_features=2048, out_features=512, bias=False, weight=IntxUnpackedToInt8Tensor(target_dtype=torch.int4, block_size=(1, 128), shape=torch.Size([512, 2048]), dtype=torch.float32, device=cpu, activation_quantization=int8_asym_per_token))
        (wv): Linear(in_features=2048, ou

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
I tokenizers:regex.cpp:40] Registering override fallback regex
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/utils.py:3895: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  return node.target(*args, **kwargs)  # type: ignore[operator]
/usr/local/lib/python3.13/dist-packages/torch/fx/interpreter.py:380: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  return target(*args, **kwargs)
/usr/lib/pytho

In [ ]:
import glob
import os

print("Current working directory:", os.getcwd())
print("All .pte files found recursively:")
for file in glob.glob('/content/**/*.pte', recursive=True):
    print(file, f"({os.path.getsize(file) / 1024**2:.2f} MB)")

Current working directory: /content
All .pte files found recursively:
/content/drive/MyDrive/waste-assistant/llama3_2_1b.pte (1337.14 MB)
/content/waste-assistant/llama3_2_1b.pte (1337.14 MB)


In [ ]:
%%bash
mkdir -p /content/drive/MyDrive/waste-assistant
cp /content/waste-assistant/*.pte /content/drive/MyDrive/waste-assistant/
cp /content/waste-assistant/llama-1b/original/tokenizer.model /content/drive/MyDrive/waste-assistant/
ls -lh /content/drive/MyDrive/waste-assistant/


total 1.4G
-rw-r--r-- 1 root root 1.4G Oct 30 07:55 llama3_2_1b.pte
-rw-r--r-- 1 root root 2.1M Oct 30 07:55 tokenizer.model


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully!")
except ValueError as e:
    if "already contain files" in str(e):
        print("Google Drive is already mounted and ready to use.")
    else:
        raise e

Google Drive is already mounted and ready to use.


In [ ]:
%%bash
DEST="/content/drive/MyDrive/Waste assistant"
mkdir -p "$DEST"
cp /content/waste-assistant/*.pte "$DEST/"
cp /content/waste-assistant/llama-1b/original/tokenizer.model "$DEST/"
ls -lh "$DEST"

total 1.4G
-rw-r--r-- 1 root root 1.4G Oct 30 08:21 llama3_2_1b.pte
-rw-r--r-- 1 root root 2.1M Oct 30 08:21 tokenizer.model


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("Drive folder exists:", os.path.exists("/content/drive/MyDrive/Waste assistant"))
print("Local files:")
for p in ["/content/waste-assistant", "/content/waste-assistant/llama-1b/original"]:
    print(p, "->", os.listdir(p) if os.path.exists(p) else "MISSING")


ValueError: Mountpoint must not already contain files

In [ ]:
%%bash
mv /content/drive /content/drive_old
ls /content


drive_old
sample_data
waste-assistant


In [ ]:
%%bash
DEST="/content/drive/MyDrive/Waste assistant"
cp /content/waste-assistant/*.pte "$DEST/"
cp /content/waste-assistant/llama-1b/original/tokenizer.model "$DEST/"
ls -lh "$DEST"


cp: cannot create regular file '/content/drive/MyDrive/Waste assistant/': No such file or directory
cp: cannot create regular file '/content/drive/MyDrive/Waste assistant/': No such file or directory
ls: cannot access '/content/drive/MyDrive/Waste assistant': No such file or directory


CalledProcessError: Command 'b'DEST="/content/drive/MyDrive/Waste assistant"\ncp /content/waste-assistant/*.pte "$DEST/"\ncp /content/waste-assistant/llama-1b/original/tokenizer.model "$DEST/"\nls -lh "$DEST"\n'' returned non-zero exit status 2.

In [ ]:
%%bash
echo "--- Drive folders:"
ls /content/drive/MyDrive | head -30
echo "--- Exported files:"
ls -lh /content/waste-assistant/*.pte
echo "--- Tokenizer:"
ls -lh /content/waste-assistant/llama-1b/original/


--- Drive folders:
--- Exported files:
-rw-r--r-- 1 root root 1.4G Oct 30 07:52 /content/waste-assistant/llama3_2_1b.pte
--- Tokenizer:
total 2.4G
-rw-r--r-- 1 root root 2.4G Oct 30 06:47 consolidated.00.pth
-rw-r--r-- 1 root root  220 Oct 30 06:46 params.json
-rw-r--r-- 1 root root 2.1M Oct 30 06:46 tokenizer.model


ls: cannot access '/content/drive/MyDrive': No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%%bash
DEST="/content/drive/MyDrive/Waste assistant"
cp /content/waste-assistant/*.pte "$DEST/"
cp /content/waste-assistant/llama-1b/original/tokenizer.model "$DEST/"
ls -lh "$DEST"


total 1.4G
-rw------- 1 root root 1.4G Oct 30 09:15 llama3_2_1b.pte
-rw------- 1 root root 2.1M Oct 30 09:15 tokenizer.model
-rw------- 1 root root  54K Oct 30 09:15  waste assistant .ipynb


In [ ]:
%%writefile /content/waste-assistant/export_config.yaml
base:
  model_class: llama3_2
  checkpoint: /content/waste-assistant/llama-1b/original/consolidated.00.pth
  params: /content/waste-assistant/llama-1b/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True


Overwriting /content/waste-assistant/export_config.yaml


In [ ]:
%%bash
DEST="/content/drive/MyDrive/Waste assistant"
cp /content/waste-assistant/*.pte "$DEST/"
ls -lh "$DEST"


total 1.4G
-rw------- 1 root root 1.4G Oct 30 09:25 llama3_2_1b.pte
-rw------- 1 root root 2.1M Oct 30 09:15 tokenizer.model
-rw------- 1 root root  56K Oct 30 09:25  waste assistant .ipynb


In [ ]:
!pip install -q executorch torchvision huggingface_hub pytorch-tokenizers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from huggingface_hub import login, snapshot_download
login()
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"],
                  local_dir="/content/waste-assistant/llama-1b")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/content/waste-assistant/llama-1b'

In [ ]:
%%writefile /content/waste-assistant/export_config.yaml
base:
  model_class: llama3_2
  checkpoint: /content/waste-assistant/llama-1b/original/consolidated.00.pth
  params: /content/waste-assistant/llama-1b/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True

Writing /content/waste-assistant/export_config.yaml


In [ ]:
%%bash
cd /content/waste-assistant
python -m executorch.extension.llm.export.export_llm --config export_config.yaml \
    +export.output_name="llama3_2_1b.pte" \
    +export.max_seq_length=1024 +export.max_context_length=1024 \
    model.use_sdpa_with_kv_cache=False \
    backend.xnnpack.enabled=True backend.xnnpack.extended_ops=True
ls -l *.pte

Process is terminated.


In [ ]:

!pip list 2>/dev/null | grep -i -E "executorch|torch"


executorch                            1.5.1
pytorch_tokenizers                    1.6.0
torch                                 2.11.0+cpu
torchao                               0.18.0
torchaudio                            2.11.0+cpu
torchcodec                            0.11.0+cpu
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.26.0+cpu


In [ ]:
!pip install -q "executorch==1.1.0"

In [ ]:
!pip list 2>/dev/null | grep -i -E "executorch|torch"


executorch                            1.1.0
pytorch_tokenizers                    1.6.0
torch                                 2.11.0+cpu
torchao                               0.15.0
torchaudio                            2.11.0+cpu
torchcodec                            0.11.0+cpu
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.26.0+cpu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%bash
ls "/content/drive/MyDrive/Waste assistant"

llama3_2_1b.pte
tokenizer.model
 waste assistant .ipynb


In [ ]:
from huggingface_hub import login, snapshot_download
login()
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"],
                  local_dir="/content/waste-assistant/llama-1b")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/content/waste-assistant/llama-1b'

In [ ]:
%%bash
ls -lh /content/waste-assistant/llama-1b/original


total 2.4G
-rw-r--r-- 1 root root 2.4G Nov  3 07:26 consolidated.00.pth
-rw-r--r-- 1 root root  220 Nov  3 07:25 params.json
-rw-r--r-- 1 root root 2.1M Nov  3 07:25 tokenizer.model


In [ ]:
%%bash
cd /content/waste-assistant
python -m executorch.extension.llm.export.export_llm --config export_config.yaml \
    +export.output_name="llama3_2_1b.pte" \
    +export.max_seq_length=1024 +export.max_context_length=1024 \
    model.use_sdpa_with_kv_cache=False \
    backend.xnnpack.enabled=True backend.xnnpack.extended_ops=True
ls -l *.pte

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cpu for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
I tokenizers:regex.cpp:40] Registering override fallback regex
Traceback (most recent call last):
  File "<frozen runpy>", line 194, in _run_module_as_main
  File "<frozen runpy>", line 114, in _get_module_details
  File "/usr/local/lib/python3.13/dist-packages/executorch/extension/llm/export/__init__.py", line 7, in <module>
    from .builder import LLMEdgeManager
  File "/usr/local/lib/python3.13/dist-packages/executorch/extension/llm/export/builder.py", line 39, in <module>
    from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_pt2e
  File "/usr/local/lib/python3.13/dist-packages/torchao/quantization/pt2e/quantize_pt2e.py", line 44, in <module>
    quantizer: Union[Quantizer, torch.ao.quantization.quantizer.quantizer.Quantizer],
                                ^^^^^^^^^^^^^^^^^^

CalledProcessError: Command 'b'cd /content/waste-assistant\npython -m executorch.extension.llm.export.export_llm --config export_config.yaml \\\n    +export.output_name="llama3_2_1b.pte" \\\n    +export.max_seq_length=1024 +export.max_context_length=1024 \\\n    model.use_sdpa_with_kv_cache=False \\\n    backend.xnnpack.enabled=True backend.xnnpack.extended_ops=True\nls -l *.pte\n'' returned non-zero exit status 2.

In [ ]:
!pip install --force-reinstall torch torchao executorch


  Using cached torchao-0.18.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (21 kB)
  Using cached executorch-1.5.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.9 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached expecttest-0.3.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached hypothesis-6.170.14-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.6 kB)
  Using cached kgb-7.3-py2.py3-none-any.whl.metadata (21 kB)
  Using cached parameterized-0.9.0-py2.py3-none-any.whl.metadata (18 kB)
  Using cached pytorch_tokenizers-1.6.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached ruamel_yaml-0.19.1-py3-none-any.whl.metadata (16 kB)
  Using cached coremltools-9.0-cp313-none-manylinux1_x86_64.whl.metadata (2.7 kB)
  Using cached scikit_learn-1.9.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached hydra_core-1.3.8-py3-none-any.whl.metadata (5.7 kB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile /content/waste-assistant/export_config.yaml
base:
  model_class: llama3_2
  checkpoint: /content/waste-assistant/llama-1b/original/consolidated.00.pth
  params: /content/waste-assistant/llama-1b/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True

Overwriting /content/waste-assistant/export_config.yaml


In [ ]:
%%bash
cd /content/waste-assistant
python -m executorch.extension.llm.export.export_llm --config export_config.yaml \
    +export.output_name="llama3_2_1b.pte" \
    +export.max_seq_length=1024 +export.max_context_length=1024 \
    model.use_sdpa_with_kv_cache=False \
    backend.xnnpack.enabled=True backend.xnnpack.extended_ops=True
ls -l *.pte

[2026-11-03 09:59:57,974][root][INFO] - Applying quantizers: []
[2026-11-03 09:59:58,284][root][INFO] - Checkpoint dtype: torch.bfloat16
quantize ('tok_embeddings', Embedding(128256, 2048)) with group_size 32, bitwidth 4
[2026-11-03 10:07:44,830][root][INFO] - Model after source transforms: Transformer(
  (tok_embeddings): QuantizedGroupEmbedding()
  (layers): ModuleList(
    (0-15): 16 x TransformerBlock(
      (attention): AttentionMHA(
        (wq): Linear(in_features=2048, out_features=2048, bias=False, weight=IntxUnpackedToInt8Tensor(target_dtype=torch.int4, block_size=(1, 128), shape=torch.Size([2048, 2048]), dtype=torch.float32, device=cpu, activation_quantization=int8_asym_per_token))
        (wk): Linear(in_features=2048, out_features=512, bias=False, weight=IntxUnpackedToInt8Tensor(target_dtype=torch.int4, block_size=(1, 128), shape=torch.Size([512, 2048]), dtype=torch.float32, device=cpu, activation_quantization=int8_asym_per_token))
        (wv): Linear(in_features=2048, ou

W1103 09:59:51.885000 40447 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1103 09:59:51.957000 40447 torch/utils/_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
I tokenizers:regex.cpp:40] Registering override fallback regex
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/utils.py:4549: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_obli

In [ ]:
%%bash
cp /content/waste-assistant/llama3_2_1b.pte "/content/drive/MyDrive/Waste assistant/"
ls -l "/content/drive/MyDrive/Waste assistant/"

cp: cannot stat '/content/waste-assistant/llama3_2_1b.pte': No such file or directory
ls: cannot access '/content/drive/MyDrive/Waste assistant/': No such file or directory


CalledProcessError: Command 'b'cp /content/waste-assistant/llama3_2_1b.pte "/content/drive/MyDrive/Waste assistant/"\nls -l "/content/drive/MyDrive/Waste assistant/"\n'' returned non-zero exit status 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile /content/waste-assistant/export_config.yaml
base:
  model_class: llama3_2
  checkpoint: /content/waste-assistant/llama-1b/original/consolidated.00.pth
  params: /content/waste-assistant/llama-1b/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True

Writing /content/waste-assistant/export_config.yaml


FileNotFoundError: [Errno 2] No such file or directory: '/content/waste-assistant/export_config.yaml'

In [ ]:
from huggingface_hub import login, snapshot_download
login()
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"],
                  local_dir="/content/waste-assistant/llama-1b")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/content/waste-assistant/llama-1b'

In [ ]:
%%writefile /content/waste-assistant/export_config.yaml
base:
  model_class: llama3_2
  checkpoint: /content/waste-assistant/llama-1b/original/consolidated.00.pth
  params: /content/waste-assistant/llama-1b/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True

Writing /content/waste-assistant/export_config.yaml


In [ ]:
%%bash
pip install -q executorch torch torchao


In [ ]:
%%bash
cd /content/waste-assistant
python -m executorch.extension.llm.export.export_llm --config export_config.yaml \
    +export.output_name="llama3_2_1b.pte" \
    +export.max_seq_length=1024 +export.max_context_length=1024 \
    model.use_sdpa_with_kv_cache=False \
    backend.xnnpack.enabled=True backend.xnnpack.extended_ops=True
ls -l *.pte

[2026-11-03 14:42:38,231][root][INFO] - Applying quantizers: []
[2026-11-03 14:42:38,371][root][INFO] - Checkpoint dtype: torch.bfloat16
quantize ('tok_embeddings', Embedding(128256, 2048)) with group_size 32, bitwidth 4
[2026-11-03 14:50:43,533][root][INFO] - Model after source transforms: Transformer(
  (tok_embeddings): QuantizedGroupEmbedding()
  (layers): ModuleList(
    (0-15): 16 x TransformerBlock(
      (attention): AttentionMHA(
        (wq): Linear(in_features=2048, out_features=2048, bias=False, weight=IntxUnpackedToInt8Tensor(target_dtype=torch.int4, block_size=(1, 128), shape=torch.Size([2048, 2048]), dtype=torch.float32, device=cpu, activation_quantization=int8_asym_per_token))
        (wk): Linear(in_features=2048, out_features=512, bias=False, weight=IntxUnpackedToInt8Tensor(target_dtype=torch.int4, block_size=(1, 128), shape=torch.Size([512, 2048]), dtype=torch.float32, device=cpu, activation_quantization=int8_asym_per_token))
        (wv): Linear(in_features=2048, ou

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
I tokenizers:regex.cpp:40] Registering override fallback regex
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/utils.py:3895: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  return node.target(*args, **kwargs)  # type: ignore[operator]
/usr/local/lib/python3.13/dist-packages/torch/fx/interpreter.py:380: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  return target(*args, **kwargs)
/usr/lib/pytho

In [ ]:
!pip install -q optimum-executorch